In [1]:
from pyspark.sql import SparkSession

## CREATING SPARK SESSION

In [2]:
spark=SparkSession.builder\
    .appName("Optimization")\
    .getOrCreate()

In [ ]:
spark

## TURN OFF AQE

In [ ]:
#aqe is always enabled from spark 3.0.0 , so we need to disable it 
spark.conf.set( "spark.sql.adaptive.enabled", "false" )
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled" , False)

In [ ]:
#checking aqe is enabled or not 
spark.conf.get("spark.sql.adaptive.enabled")

## READING DATA FROM MY SPARK 

In [ ]:
df=spark.read.format("csv").option("header" , True).option("inferSchema" , True).load("BigMart Sales.csv")
#inferSchema makes the spark to understand the schema from the data and 

In [ ]:
display(df)

In [ ]:
df.show()

In [ ]:
df.count()

In [ ]:
df.rdd.getNumPartitions()

## CHANGING DEFAULT PARTITION SIZE TO 128KB

By default the partition size the spark fixes is 128 mb , so we are getting one partition only .
So to make sense of partitions , we are changing the default size of partition bytes

## Our data is of 849kb

In [ ]:
spark.conf.set("spark.sql.files.maxPartitionBytes" , 131072)
#128kb=131072bytes

In [ ]:
df=spark.read.format("csv").option("headers" , True).option("inferSchema" , True).load("BigMart Sales.csv")

In [ ]:
df.rdd.getNumPartitions()
#849kb/128kb =~ 6.63
# So we got 7 partitions 

## Changing back to default 128 mb

In [ ]:
spark.conf.set("spark.sql.files.maxPartitionBytes" , 134217728)

## DISCOVERING REPARITION
without changing the default partition size , we are only changing the number of partitions using the repartition. 

In [ ]:
df=df.repartition(10)

In [ ]:
df.rdd.getNumPartitions()

## GETTING SPARK PARTITION INFO

In [ ]:
from pyspark.sql.functions import spark_partition_id

In [ ]:
Pdf=df.withColumn("partition_id" , spark_partition_id())
#spark_partition_id gives the partition number where the data is present 
Pdf.show()

In [ ]:
Pdf.filter(Pdf["partition_id"]==1).select("*").show()
# getting the records which are present in partition id 1 

## WRITING DATA INTO FILES 

In [ ]:
df.write.mode("append").parquet("data")

In [ ]:
# df.write.format("noop").mode("overwrite").save()
#if you dont want to write files but want to write it in some kind of invisible place and just see in the ui . You can go with this . 